In [4]:
import pandas as pd

columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week",
    "native_country", "income"
]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

df = pd.read_csv(url, header=None, names=columns, skipinitialspace=True)

df.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [5]:
df.replace("?", pd.NA, inplace=True)
df.dropna(inplace=True)
df.shape

(30162, 15)

 Import libraries

In [9]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 131.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 167.9 MB/s eta 0:00:0000:01


In [10]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef,
    confusion_matrix
)

import joblib


In [13]:
# Encode Categorical Columns

le = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])


In [14]:
# Encode Categorical Columns

le = LabelEncoder()

for col in df.select_dtypes(include="object").columns:
    df[col] = le.fit_transform(df[col])


In [15]:
# Split Features & Target

X = df.drop("income", axis=1)
y = df["income"]


In [16]:
# Train–Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [17]:
# Feature Scaling (Important for KNN & LR)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [18]:
# Initialize All Models

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

Train Models & Evaluate Metrics

In [19]:
# Evaluation Function

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred)
    })

    # Save model
    joblib.dump(model, f"{name.replace(' ', '_').lower()}.pkl")


In [20]:
# View Results Table

results_df = pd.DataFrame(results)
results_df


,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.817504,0.850060,0.713525,0.446072,0.548955,0.461262
1,Decision Tree,0.806067,0.740037,0.610963,0.608522,0.609740,0.480717
2,KNN,0.818996,0.849777,0.652985,0.582557,0.615764,0.499267
3,Naive Bayes,0.797779,0.849760,0.698592,0.330226,0.448463,0.379756
4,Random Forest,0.854136,0.902419,0.743349,0.632490,0.683453,0.592730


XGBoost

In [ ]:
%pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 MB 161.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 MB 106.9 MB/s eta 0:00:0000:0100:01


In [22]:
import xgboost
print(xgboost.__version__)


3.1.3


In [26]:
# Train XGBoost

from xgboost import XGBClassifier

xgb = XGBClassifier(
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=42
)

xgb.fit(X_train, y_train)

y_pred = xgb.predict(X_test)
y_prob = xgb.predict_proba(X_test)[:, 1]

results.append({
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test, y_pred),
    "AUC": roc_auc_score(y_test, y_prob),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "MCC": matthews_corrcoef(y_test, y_pred)
})

joblib.dump(xgb, "xgboost.pkl")


/home/cloud/anaconda3/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [19:34:01] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


['xgboost.pkl']

In [27]:
pd.DataFrame(results)

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.817504,0.850060,0.713525,0.446072,0.548955,0.461262
1,Decision Tree,0.806067,0.740037,0.610963,0.608522,0.609740,0.480717
2,KNN,0.818996,0.849777,0.652985,0.582557,0.615764,0.499267
3,Naive Bayes,0.797779,0.849760,0.698592,0.330226,0.448463,0.379756
4,Random Forest,0.854136,0.902419,0.743349,0.632490,0.683453,0.592730
5,XGBoost,0.861595,0.920449,0.763636,0.643142,0.698229,0.613069
6,XGBoost,0.861595,0.920449,0.763636,0.643142,0.698229,0.613069
7,XGBoost,0.861595,0.920449,0.763636,0.643142,0.698229,0.613069
